# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:

TASK_ID = "task054"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task054.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / f"{TASK_ID}.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / f"{TASK_ID}_static_model_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
with TASK_JSON.open('r') as f:
    task=json.load(f)
print(TASK_ID, len(task.get('train', [])), len(task.get('test', [])), len(task.get('arc-gen', [])))


task054 3 1 262


In [6]:

def grid_to_tensor(grid, h=H, w=W, ch=CH, full_background=False):
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    if full_background:
        x[0,0,:,:]=1.0
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            if full_background:
                x[0,:,r,c]=0.0
            x[0,int(v),r,c]=1.0
    return x

def tensor_to_grid(y, h, w):
    return y[0,:,:h,:w].argmax(axis=0).astype(int).tolist()

def expected_tensor(ex):
    return grid_to_tensor(ex['output'], full_background=False)


In [7]:

class Task054LegendRayPropagation(nn.Module):
    """Legend-guided ray/local-stencil propagation inside active canvases.

    Rule model:
    1. Infer background as most frequent color and canvas as second-most frequent color.
    2. Infer marker color from non-background/non-canvas pixels adjacent to canvas regions.
    3. Treat the off-canvas legend as a symbolic 5x5 stencil family.
    4. Propagate horizontal/vertical rays through the connected canvas segment containing each marker;
       apply local side/diagonal stencil masks; remove the off-canvas legend.
    """
    def __init__(self):
        super().__init__()
        rr=torch.arange(30,dtype=torch.float32).view(1,1,30,1).expand(1,1,30,30)
        cc=torch.arange(30,dtype=torch.float32).view(1,1,1,30).expand(1,1,30,30)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
        self.register_buffer('big', torch.tensor(1000.0, dtype=torch.float32))
        self.register_buffer('kadj', torch.tensor([[[[0.,1.,0.],[1.,0.,1.],[0.,1.,0.]]]], dtype=torch.float32))
        self.register_buffer('kh', torch.tensor([[[[0.,0.,0.],[1.,1.,1.],[0.,0.,0.]]]], dtype=torch.float32))
        self.register_buffer('kv', torch.tensor([[[[0.,1.,0.],[0.,1.,0.],[0.,1.,0.]]]], dtype=torch.float32))
        kd=torch.zeros(1,1,5,5)
        for dr in [-1,1]:
            for dc in [-1,1]:
                kd[0,0,2-dr,2-dc]=1.0
        self.register_buffer('kdiag', kd)
        kvs=torch.zeros(1,1,5,5)
        for dr in [-1,0,1]:
            for dc in [-1,1]:
                kvs[0,0,2-dr,2-dc]=1.0
        self.register_buffer('kvside', kvs)
        khs=torch.zeros(1,1,5,5)
        for dr in [-1,1]:
            for dc in [-1,0,1]:
                khs[0,0,2-dr,2-dc]=1.0
        self.register_buffer('khside', khs)

    def h_flood(self, centers, allowed):
        m=centers*allowed
        for _ in range(30):
            m=(F.conv2d(m,self.kh,padding=1)>0.5).float()*allowed
        return m*(1-centers)

    def v_flood(self, centers, allowed):
        m=centers*allowed
        for _ in range(30):
            m=(F.conv2d(m,self.kv,padding=1)>0.5).float()*allowed
        return m*(1-centers)

    def bbox_hw(self, mask):
        rmin=(mask*self.rr+(1-mask)*self.big).amin(dim=(2,3),keepdim=True)
        rmax=(mask*self.rr).amax(dim=(2,3),keepdim=True)
        cmin=(mask*self.cc+(1-mask)*self.big).amin(dim=(2,3),keepdim=True)
        cmax=(mask*self.cc).amax(dim=(2,3),keepdim=True)
        return rmax-rmin+1, cmax-cmin+1

    def forward(self,x):
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        counts=(x*active).sum(dim=(2,3))
        bgmax=counts.max(dim=1,keepdim=True).values
        bg_oh=(torch.abs(counts-bgmax)<0.5).float()
        counts2=counts*(1-bg_oh)
        canvas_max=counts2.max(dim=1,keepdim=True).values
        canvas_oh=(torch.abs(counts2-canvas_max)<0.5).float()
        bg=(x*bg_oh.view(1,10,1,1)).sum(dim=1,keepdim=True)
        canvas=(x*canvas_oh.view(1,10,1,1)).sum(dim=1,keepdim=True)
        adj=(F.conv2d(canvas,self.kadj,padding=1)>0.5).float()
        suppress=(1-bg_oh)*(1-canvas_oh)
        scores=(x*adj).sum(dim=(2,3))*suppress
        smax=scores.max(dim=1,keepdim=True).values
        marker_oh=(torch.abs(scores-smax)<0.5).float()*suppress*(smax>0.5).float()
        marker=(x*marker_oh.view(1,10,1,1)).sum(dim=1,keepdim=True)
        centers=marker*adj
        other=torch.clamp(active-bg-canvas,0,1)
        legend=torch.clamp(other*(1-centers),0,1)
        allowed=torch.clamp(canvas+centers,0,1)
        lcounts=(x*legend).sum(dim=(2,3))
        mcount=(lcounts*marker_oh).sum(dim=1,keepdim=True)
        nonmarker=(1-bg_oh)*(1-canvas_oh)*(1-marker_oh)
        both_oh=(torch.abs(lcounts-8.0)<0.5).float()*nonmarker
        count4_oh=(torch.abs(lcounts-4.0)<0.5).float()*nonmarker
        side_oh=(torch.abs(lcounts-6.0)<0.5).float()*nonmarker
        color_masks=x*legend
        heights=[]; widths=[]
        for col in range(10):
            h,w=self.bbox_hw(color_masks[:,col:col+1])
            heights.append(h); widths.append(w)
        heights=torch.cat(heights,dim=1).view(1,10,1,1)
        widths=torch.cat(widths,dim=1).view(1,10,1,1)
        c4=count4_oh.view(1,10,1,1)
        v4=(heights>widths).float()*c4
        h4=(widths>heights).float()*c4
        hray_oh=torch.clamp(both_oh.view(1,10,1,1)+h4,0,1)
        vray_oh=torch.clamp(both_oh.view(1,10,1,1)+v4,0,1)
        diag_flag=(mcount>1.5).float()
        diag=(F.conv2d(centers,self.kdiag,padding=2)>0.5).float()*allowed*diag_flag.view(1,1,1,1)
        vside=(F.conv2d(centers,self.kvside,padding=2)>0.5).float()*allowed*((v4.sum(dim=1,keepdim=True)>0.5).float())
        hside=(F.conv2d(centers,self.khside,padding=2)>0.5).float()*allowed*((h4.sum(dim=1,keepdim=True)>0.5).float())
        hseg=self.h_flood(centers,allowed)
        vseg=self.v_flood(centers,allowed)
        draw=[]
        for col in range(10):
            col_marker=marker_oh[:,col:col+1].view(1,1,1,1)
            col_side=side_oh[:,col:col+1].view(1,1,1,1)
            col_hr=hray_oh[:,col:col+1]
            col_vr=vray_oh[:,col:col+1]
            m=torch.clamp(centers*col_marker + diag*col_marker + vside*col_side + hside*col_side + hseg*col_hr + vseg*col_vr,0,1)
            draw.append(m)
        draw_any=torch.clamp(sum(draw),0,1)
        channels=[]
        for col in range(10):
            base=x[:,col:col+1]*(1-legend) + bg_oh[:,col:col+1].view(1,1,1,1)*legend
            channels.append(torch.clamp(base*(1-draw_any)+draw[col],0,1)*active)
        return torch.cat(channels,dim=1)*active

model=Task054LegendRayPropagation().eval()


In [8]:

dummy = torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=['input'], output_names=['output'],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
print('ONNX:', ONNX_PATH, 'bytes:', ONNX_PATH.stat().st_size)


/tmp/ipykernel_16/1118339530.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task054_static_model_onnx/task054.onnx bytes: 172589


In [9]:

def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
summary_static = {
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
}
print(json.dumps(summary_static, indent=2)[:4000])
assert summary_static['input_shape'] == [1,10,30,30]
assert summary_static['output_shape'] == [1,10,30,30]
assert summary_static['onnx_size_bytes'] < 1_400_000
assert not summary_static['forbidden_ops']


{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 172589,
  "ops": {
    "Constant": 486,
    "ReduceSum": 10,
    "Greater": 71,
    "Cast": 77,
    "Mul": 220,
    "ReduceMax": 23,
    "Sub": 44,
    "Abs": 6,
    "Less": 6,
    "Reshape": 38,
    "Conv": 64,
    "Clip": 26,
    "Add": 122,
    "Slice": 70,
    "ReduceMin": 20,
    "Concat": 3
  },
  "forbidden_ops": []
}


In [10]:

sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples, full_background=False):
    ok=0; bad=[]; outside_zero_ok=0; active_exact_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex['input'], full_background=full_background)
        y=sess.run(None, {'input':x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=expected_tensor(ex)
        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1,keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_exact_ok += bool(np.array_equal(pred*active, exp*active))
    return {'ok':ok, 'total':len(examples), 'bad_first10':bad[:10], 'outside_zero_ok':outside_zero_ok, 'active_exact_ok':active_exact_ok}

rng=random.Random(0)
inds=list(range(len(task.get('arc-gen', []))))
rng.shuffle(inds)
hold=[task['arc-gen'][i] for i in inds[:math.ceil(0.6*len(inds))]] if inds else []
validation={
    'strict_zero_padding': {
        'train': validate_examples(task['train'], False),
        'test': validate_examples(task['test'], False),
        'arc_gen_60pct_holdout': validate_examples(hold, False) if hold else None,
    },
    'full_background_padding_diagnostic': {
        'train': validate_examples(task['train'], True),
        'test': validate_examples(task['test'], True),
    }
}
summary={'task_id':TASK_ID, 'model_class':model.__class__.__name__, **summary_static, 'validation':validation}
json.dump(summary, open(SUMMARY_PATH,'w'), indent=2)
print(json.dumps(summary, indent=2)[:5000])
assert validation['strict_zero_padding']['train']['ok'] == validation['strict_zero_padding']['train']['total']
assert validation['strict_zero_padding']['test']['ok'] == validation['strict_zero_padding']['test']['total']
if hold:
    assert validation['strict_zero_padding']['arc_gen_60pct_holdout']['ok'] == validation['strict_zero_padding']['arc_gen_60pct_holdout']['total']


{
  "task_id": "task054",
  "model_class": "Task054LegendRayPropagation",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 172589,
  "ops": {
    "Constant": 486,
    "ReduceSum": 10,
    "Greater": 71,
    "Cast": 77,
    "Mul": 220,
    "ReduceMax": 23,
    "Sub": 44,
    "Abs": 6,
    "Less": 6,
    "Reshape": 38,
    "Conv": 64,
    "Clip": 26,
    "Add": 122,
    "Slice": 70,
    "ReduceMin": 20,
    "Concat": 3
  },
  "forbidden_ops": [],
  "validation": {
    "strict_zero_padding": {
      "train": {
        "ok": 3,
        "total": 3,
        "bad_first10": [],
        "outside_zero_ok": 3,
        "active_exact_ok": 3
      },
      "test": {
        "ok": 1,
        "total": 1,
        "bad_first10": [],
        "outside_zero_ok": 1,
        "active_exact_ok": 1
      },
      "arc_gen_60pct_holdout": {
        "ok": 158,
        "total": 158,
        "bad_first10": [],
        "outside_zero_ok"

In [11]:

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']


Wrote: /kaggle/working/submission.zip
Zip contents: ['task054.onnx']
